# Signals in the Noise — RCA weekly freight forecasting
**Production notebook.** Weekly training on 10 business-unit groups, forecasts aggregated to month / quarter /
year. Models (endogenous only, no macroeconomic inputs): seasonal-naive, ETS, SARIMA, Prophet (holiday-aware),
XGBoost (+ optional LSTM for mineral oil).

### Requirements
```bash
pip install pandas numpy matplotlib statsmodels prophet xgboost holidays torch
```
### Configuration
Point `SRC` at the master file `MasterPredictions200501-202604.xlsx`.

In [ ]:
import os, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
import holidays as hol_lib

# ---- reproducibility: fix all random seeds ----
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass

PROJECT  = os.path.expanduser("~/Documents/EMBA/MasterThesis")
SRC      = os.path.join(PROJECT, "Data", "MasterPredictions200501-202604.xlsx")
HOLD_W   = 104          # weeks held out (2 years)
PER      = 52           # weekly annual season

UNIT2GROUP = {
 "Agriculture":"agriculture","Automotive":"automotive","Build.M./Constr.Log.":"buildingmaterials",
 "Chemicals":"chemicals","Consumer Goods":"consumergoods","Environmental":"environmental",
 "Mineral Oil":"mineraloil","Paper":"paper","Wood":"wood",
 "Voest":"montan","Steel":"montan","Energy/Raw Materials":"montan",
}  # excluded: Intermodal (UKV/ROLA), Empty Wagons, residual/overhead/n.def./BU MAC
GROUPS = ["agriculture","automotive","buildingmaterials","chemicals","consumergoods",
          "environmental","mineraloil","montan","paper","wood"]
print("config set")

## 1  Build the weekly panel from the master file
VJA = year of shipment, VKW = ISO week of shipment, Geschäftseinheit = unit, TOKM = tonne-kilometres.
The valuation fields (BWJAMO) and VMO are ignored. Incomplete tail weeks of 2026 (and the spurious week 53)
are dropped.

In [ ]:
df = pd.read_excel(SRC, engine="openpyxl", usecols=["VJA","VKW","Geschäftseinheit","TOKM"])
df["group"] = df["Geschäftseinheit"].map(UNIT2GROUP)
df = df.dropna(subset=["group"])
df = df[~((df.VJA == 2026) & (df.VKW > 20))]                 # drop incomplete 2026 tail + stray week 53
df["TTOKM"] = df["TOKM"] / 1000.0                            # -> thousand tonne-km

g = df.groupby(["group","VJA","VKW"], as_index=False)["TTOKM"].sum()
def iso(y, w):
    try: return pd.Timestamp.fromisocalendar(int(y), int(w), 1)
    except: return pd.NaT
g["date"] = [iso(y, w) for y, w in zip(g.VJA, g.VKW)]
W = g.dropna(subset=["date"]).sort_values(["group","date"])

def wser(grp):
    s = (W.groupby("date")["TTOKM"].sum() if grp=="TOTAL"
         else W[W.group==grp].set_index("date")["TTOKM"])
    return s[~s.index.duplicated()].sort_index()

print("weeks per group:", W.groupby("group").size().to_dict())
print("range:", W.date.min().date(), "->", W.date.max().date())

## 2  Helpers: calendar feature, Fourier terms, metrics, aggregation
For the long 52-week season we use **deterministic Fourier terms** (date-derived, not macro data) in the SARIMA
estimator — the standard, fast way to model annual seasonality in weekly data (a full seasonal_order with s=52
is impractically slow). `hol_week` provides the public-holiday count used only by Prophet.

In [ ]:
def hol_week(idx):
    at = hol_lib.Austria()
    out=[]
    for ts in idx:
        days=[ (ts+pd.Timedelta(days=d)) for d in range(7)]
        out.append(sum(1 for d in days if d in at))
    return np.array(out, float)

def fourier(idx, K=5, period=PER):
    w = idx.isocalendar().week.values.astype(float)
    cols=[]
    for k in range(1,K+1):
        cols += [np.sin(2*np.pi*k*w/period), np.cos(2*np.pi*k*w/period)]
    return np.column_stack(cols)

def mape(a,f): a,f=np.asarray(a,float),np.asarray(f,float); return np.mean(np.abs((a-f)/a))*100
def agg(s,fr): return s.groupby(s.index.to_period(fr)).sum()
def agg_mape(actual_w, fc_w, fr):
    a=agg(actual_w,fr); f=agg(pd.Series(fc_w,index=actual_w.index),fr); ix=a.index.intersection(f.index)
    return mape(a[ix], f[ix])
print("helpers ready")

## 3  Models (weekly)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def f_snaive(tr,h,fidx):
    v=list(tr.values); return np.array([v[-PER] if i<PER else v[-PER+(i%PER)] for i in range(h)])

def f_ets(tr,h,fidx):
    return ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=PER).fit().forecast(h).values

def f_sarima(tr,h,fidx):
    # SARIMA(2,1,1) + deterministic Fourier(K=5) terms for the annual cycle (no macro inputs).
    # statsmodels' SARIMAX class is just the estimator here; the Fourier terms are date-derived, not exogenous data.
    Ftr=fourier(tr.index); Ff=fourier(fidx)
    fit=SARIMAX(tr, exog=Ftr, order=(2,1,1), enforce_stationarity=False,
                enforce_invertibility=False).fit(disp=False)
    return fit.forecast(h, exog=Ff).values

def f_prophet(tr,h,fidx):
    from prophet import Prophet
    d=pd.DataFrame({"ds":tr.index,"y":tr.values})
    mdl=Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=0.05)
    mdl.add_country_holidays(country_name="AT")   # deterministic calendar (holidays), not macro
    mdl.fit(d)
    return mdl.predict(pd.DataFrame({"ds":fidx}))["yhat"].values

def f_xgboost(tr,h,fidx):
    from xgboost import XGBRegressor
    y=tr.values.astype(float); idx=tr.index
    def fr(ti,fi,hist):
        return [ti,hist[-1],hist[-2],hist[-4],hist[-PER],np.mean(hist[-4:]),np.mean(hist[-PER:])] + \
               [int(fi.month==k) for k in range(1,13)]
    X=[fr(ti,idx[ti],y[:ti]) for ti in range(PER+4,len(y))]; Y=y[PER+4:]
    mdl=XGBRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,subsample=0.9,
                     random_state=SEED,n_jobs=1).fit(np.array(X),Y)
    hist=list(y); out=[]
    for i in range(h):
        row=np.array(fr(len(y)+i,fidx[i],np.array(hist))).reshape(1,-1)
        p=float(mdl.predict(row)[0]); out.append(p); hist.append(p)
    return np.array(out)

MODELS={"SeasonalNaive":f_snaive,"ETS":f_ets,"SARIMA":f_sarima,
        "Prophet":f_prophet,"XGBoost":f_xgboost}   # endogenous only; SARIMAX (macro) dropped
print("models defined")

## 4  Back-test: weekly forecast, error at weekly/monthly/quarterly/annual

In [ ]:
def backtest(g):
    s=wser(g).dropna(); tr=s.iloc[:-HOLD_W]; te=s.iloc[-HOLD_W:]
    rec={}
    for name,fn in MODELS.items():
        try:
            fc=fn(tr,HOLD_W,te.index)
            rec[name]={"weekly":mape(te.values,fc),
                       "monthly":agg_mape(te,fc,"M"),
                       "quarterly":agg_mape(te,fc,"Q"),
                       "annual":agg_mape(te,fc,"Y")}
        except Exception as e:
            rec[name]={"weekly":np.nan,"monthly":np.nan,"quarterly":np.nan,"annual":np.nan}
            print(g,name,"failed:",e)
    return rec

results={g:backtest(g) for g in ["TOTAL"]+GROUPS}
tbl=pd.DataFrame({g:{m:results[g][m]["monthly"] for m in MODELS} for g in results}).T.round(1)
print("Monthly-aggregated MAPE %:"); display(tbl)
# aggregation benefit for the total
pd.DataFrame(results["TOTAL"]).T.round(2)

## 5  Experiment — training-history length (weekly)

In [ ]:
for g in ["TOTAL","montan","mineraloil","wood"]:
    s=wser(g).dropna(); te=s.iloc[-HOLD_W:]; hist=s.iloc[:-HOLD_W]; row={}
    for yrs in [5,10,15,20]:
        tr=hist.iloc[-yrs*52:]
        row[yrs]=round(agg_mape(te, f_sarima(tr,HOLD_W,te.index), "M"),2)
    print(g, row)

## 6  LSTM spotlight — mineral oil (optional, needs torch)

In [ ]:
def f_lstm(tr,h,fidx,lookback=52,epochs=200):
    import torch, torch.nn as nn
    torch.manual_seed(SEED); np.random.seed(SEED)   # reproducible weight init
    y=tr.values.astype("float32"); mu,sd=y.mean(),y.std(); z=(y-mu)/sd
    X=np.array([z[i:i+lookback] for i in range(len(z)-lookback)])
    Y=np.array([z[i+lookback] for i in range(len(z)-lookback)])
    Xt=torch.tensor(X).unsqueeze(-1); Yt=torch.tensor(Y).unsqueeze(-1)
    class Net(nn.Module):
        def __init__(s): super().__init__(); s.l=nn.LSTM(1,48,batch_first=True); s.f=nn.Linear(48,1)
        def forward(s,x): o,_=s.l(x); return s.f(o[:,-1])
    net=Net(); opt=torch.optim.Adam(net.parameters(),0.01); L=nn.MSELoss()
    for _ in range(epochs): opt.zero_grad(); L(net(Xt),Yt).backward(); opt.step()
    seq=list(z[-lookback:]); out=[]
    for _ in range(h):
        xi=torch.tensor(np.array(seq[-lookback:],dtype="float32")).reshape(1,lookback,1)
        p=float(net(xi).item()); out.append(p); seq.append(p)
    return np.array(out)*sd+mu

s=wser("mineraloil").dropna(); tr=s.iloc[:-HOLD_W]; te=s.iloc[-HOLD_W:]
try:
    fc=f_lstm(tr,HOLD_W,te.index)
    print("mineral oil LSTM — monthly MAPE: %.2f%%" % agg_mape(te,fc,"M"))
except ImportError:
    print("Install torch to run the LSTM cell.")

## 7  Export results

In [ ]:
RESULTS_DIR=os.path.join(PROJECT,"results_final"); os.makedirs(RESULTS_DIR,exist_ok=True)
out=pd.DataFrame({g:{m:results[g][m]["monthly"] for m in MODELS} for g in results}).T.round(2)
out.to_csv(os.path.join(RESULTS_DIR,"weekly_model_metrics_monthly.csv"))
# champion model per group (monthly back-test)
champ=pd.DataFrame([(g,min(MODELS,key=lambda k:results[g][k]["monthly"]),
                     round(min(results[g][k]["monthly"] for k in MODELS),2)) for g in results],
                   columns=["group","champion_model","monthly_MAPE"])
champ.to_csv(os.path.join(RESULTS_DIR,"champion_per_group.csv"),index=False)
print("written to", RESULTS_DIR); champ